# ScreamingFace · compose Fusions

Build reusable answer recipes without running them. A Fusion can call one model directly or
combine the answers of other Fusions.

Construction and `.url4` compilation are entirely network-free. This notebook needs no Docker,
provider credentials, or benchmark data.

## 1 · Start with model IDs

In [ ]:
import screamingface as sf

frontier_fusion = sf.Fusion(
    "frontier-trio",
    inputs=[
        "codex/gpt-5.5",
        "gemini/2.5-flash",
        "claude/sonnet-4.6",
    ],
    reducer=sf.reducers.MajorityVote(),
)

frontier_fusion

Use model-ID strings by default. Each string is shorthand for an anonymous atomic
Fusion using the minimal prompt `Answer the question.`. The list preserves stable input order.

`MajorityVote()` selects the most common exact answer. Ties resolve by stable member order, and the
reducer makes no additional model call.

## 2 · Inspect the public authoring values

In [ ]:
{
    "inputs": frontier_fusion.inputs,
    "model_ids": frontier_fusion.model_ids,
    "reducer": frontier_fusion.reducer,
    "url4": frontier_fusion.url4,
}

These four values are enough to inspect or share the definition:

- `inputs` preserves the concise strings and explicit input Fusions;
- `model_ids` provides the ordered route IDs alone;
- `reducer` is the immutable reduction strategy; and
- `url4` is the canonical recipe with its `$question` input still unbound.

Reading any of them remains network-free.

## 3 · Configure an atomic Fusion explicitly

In [ ]:
scientist = sf.Fusion(
    "gemini-scientist",
    model="gemini/2.5-flash",
    prompt="Check the scientific reasoning and answer directly.",
    params={"temperature": 0.2, "max_tokens": 512},
)

specialist_fusion = sf.Fusion(
    "specialist-pair",
    inputs=["codex/gpt-5.5", scientist],
    reducer=sf.reducers.MajorityVote(),
)

specialist_fusion.inputs

An atomic Fusion owns one model call and may define its own prompt and parameters.
Use it anywhere an input needs more than the string shorthand.

Each parameter value must be a string, integer, finite float, or boolean. `tools` is reserved: tool
requirements belong to the benchmark so ScreamingFace can apply them consistently to every
answer-producing member.

## 4 · The same model can be more than one member

In [ ]:
SELF_MODEL = "claude/sonnet-4.6"

sample_1 = sf.Fusion(
    "claude-sample-1",
    model=SELF_MODEL,
    prompt="Solve independently and favor precise evidence.",
    params={"temperature": 0.2},
)
sample_2 = sf.Fusion(
    "claude-sample-2",
    model=SELF_MODEL,
    prompt="Challenge the obvious answer and check alternatives.",
    params={"temperature": 0.8},
)

self_fusion = sf.Fusion(
    "claude-independent-samples",
    inputs=[sample_1, sample_2],
    reducer=sf.reducers.Model(
        model="codex/gpt-5.5",
        prompt="Synthesize the strongest supported answer from the panel.",
        params={"temperature": 0.0, "max_tokens": 512},
    ),
)

self_fusion.model_ids

Members are positions in the panel, not unique model names. Repeating a route is
therefore valid: the two Claude members become separate ordered requests with separate prompts and
parameters.

`reducers.Model(...)` makes one additional model call. It receives the original question plus
every labeled input answer and synthesizes the Fusion answer. Reusing the same atomic Fusion object
elsewhere reuses its answer node; constructing another one creates an independent sample.

## 5 · Inspect the self-Fusion recipe

In [ ]:
{
    "inputs": self_fusion.inputs,
    "model_ids": self_fusion.model_ids,
    "reducer": self_fusion.reducer,
    "url4": self_fusion.url4,
}

The recipe records the ordered member calls and the reducer call, but it still does
nothing until a concrete question is supplied through execution.

## What construction does not prove

Fusion construction validates the local value shape only. It cannot establish that a configured
engine advertises these routes, supports a benchmark's required tools, or has working provider
credentials. That compatibility is checked when execution begins.

## Recap

- prefer model-ID strings for ordinary anonymous inputs;
- use an atomic Fusion for a named prompt or parameters;
- input order is stable and model IDs may repeat;
- `MajorityVote()` is deterministic and adds no model call;
- `Model(...)` makes one additional synthesis call; and
- construction plus `.url4` inspection are network-free.

Continue to the quickstart to evaluate a Fusion or to the custom-benchmark guide to define your own
cases and scoring contract.